In [1]:
%pip install torch numpy pandas scikit-learn matplotlib seaborn scipy statsmodels

# Optional — GPU-accelerated kernel (requires CUDA 11.6+)
%pip install mamba-ssm causal-conv1d


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
      /private/var/folders/11/1g_3qz7x02s7lddn58_9fr2h0000gn/T/pip-build-env-p8jl716p/overlay/lib/python3.13/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
        warn(
      /private/var/folders/11/1g_3qz7x02s7lddn58_9fr2h0000gn/T/pip-build-env-p8jl716p/overlay/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: 

In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from copy        import deepcopy
from scipy       import stats
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

In [ ]:
INPUT_FILE  = "merged_features_weekly.csv"
OUTPUT_DIR  = "."

TARGET      = "target_price"
TEST_WEEKS  = 52
LOOKBACK    = 52       # input window length (weeks)
HORIZON     = 1        # steps ahead per forward pass (walk-forward)

FEATURE_COLS = [
    "api_energy_and_lubricants",
    "api_fertilisers_and_soil_improvers",
    "api_plant_protection_products",
    "api_fresh_fruit",
    "api_fresh_vegetables",
    "fuel_petrol_price",
    "fuel_diesel_price",
    "sppi_road_freight",
    "shock_2021q4_2023q1",
    "post_shock",
    "week_sin",
    "week_cos",
]

# Model hyperparameters
D_MODEL  = 64    # model (embedding) dimension
D_STATE  = 16    # SSM state dimension N
D_CONV   = 4     # local conv width inside each Mamba block
EXPAND   = 2     # inner dimension = D_MODEL * EXPAND
N_LAYERS = 3     # number of stacked Mamba blocks
DROPOUT  = 0.1

# Training
BATCH_SIZE = 32
EPOCHS     = 150
LR         = 1e-3
PATIENCE   = 20
SEED       = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
try:
    from mamba_ssm import Mamba as MambaKernel
    _USE_OFFICIAL_MAMBA = True
    print("✓ mamba-ssm found — using optimised CUDA kernel")
except ImportError:
    _USE_OFFICIAL_MAMBA = False
    print("ℹ mamba-ssm not installed — using pure-PyTorch SSM fallback")


class SelectiveSSM(nn.Module):
    """
    Pure-PyTorch implementation of the selective state-space scan
    (Mamba core). Matches the paper's equations:
        h_t = A_bar * h_{t-1} + B_bar * x_t
        y_t = C * h_t
    where A_bar, B_bar, C are input-dependent (selective).
    """
    def __init__(self, d_model: int, d_state: int = 16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Input-dependent projections
        self.x_proj   = nn.Linear(d_model, d_state * 2 + d_model, bias=False)
        self.dt_proj  = nn.Linear(d_model, d_model, bias=True)

        # Learnable log-A (diagonal, real, initialised as in the paper)
        A_log = torch.arange(1, d_state + 1, dtype=torch.float32).log()
        self.A_log = nn.Parameter(A_log.repeat(d_model, 1))   # (d, N)
        self.D     = nn.Parameter(torch.ones(d_model))         # skip connection

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, d_model) → (B, L, d_model)"""
        B, L, d = x.shape
        N = self.d_state

        # Project x → dt, B_ssm, C_ssm
        xz    = self.x_proj(x)                                  # (B,L, N*2+d)
        B_ssm = xz[..., :N]                                     # (B,L,N)
        C_ssm = xz[..., N:2*N]                                  # (B,L,N)
        dt    = F.softplus(self.dt_proj(xz[..., 2*N:]))         # (B,L,d)

        # Discretise A: A_bar = exp(dt * A)
        A     = -self.A_log.exp()                                # (d,N)
        # dt: (B,L,d), A: (d,N) → (B,L,d,N)
        A_bar = torch.exp(dt.unsqueeze(-1) * A)                  # (B,L,d,N)
        # B_bar = dt * B_ssm (broadcast)
        B_bar = dt.unsqueeze(-1) * B_ssm.unsqueeze(2)           # (B,L,d,N)

        # Sequential scan (can be parallelised; kept simple for clarity)
        h      = torch.zeros(B, d, N, device=x.device, dtype=x.dtype)
        ys     = []
        for t in range(L):
            h  = A_bar[:, t] * h + B_bar[:, t] * x[:, t].unsqueeze(-1)
            y  = (h * C_ssm[:, t].unsqueeze(1)).sum(-1)          # (B,d)
            ys.append(y)
        y_seq = torch.stack(ys, dim=1)                           # (B,L,d)
        return y_seq + self.D * x


class MambaBlock(nn.Module):
    """
    Full Mamba block:  LayerNorm → expand → conv1d → SSM → gate → project
    Uses official mamba-ssm kernel if available, otherwise SelectiveSSM.
    """
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm  = nn.LayerNorm(d_model)
        d_inner    = int(d_model * expand)

        if _USE_OFFICIAL_MAMBA:
            self.mamba = MambaKernel(
                d_model = d_model,
                d_state = d_state,
                d_conv  = d_conv,
                expand  = expand,
            )
            self._use_official = True
        else:
            self.in_proj   = nn.Linear(d_model, d_inner * 2, bias=False)
            self.conv1d    = nn.Conv1d(d_inner, d_inner, d_conv,
                                       padding=d_conv - 1, groups=d_inner, bias=True)
            self.ssm       = SelectiveSSM(d_inner, d_state)
            self.out_proj  = nn.Linear(d_inner, d_model, bias=False)
            self._use_official = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, d_model)"""
        residual = x
        x = self.norm(x)

        if self._use_official:
            return self.mamba(x) + residual

        # Manual path
        xz       = self.in_proj(x)                              # (B,L, 2*d_inner)
        x_in, z  = xz.chunk(2, dim=-1)                         # each (B,L,d_inner)
        # Causal conv1d
        x_conv   = self.conv1d(x_in.transpose(1, 2))           # (B,d_inner, L+pad)
        x_conv   = x_conv[..., :x_in.shape[1]].transpose(1, 2) # (B,L,d_inner)
        x_conv   = F.silu(x_conv)
        y        = self.ssm(x_conv)
        y        = y * F.silu(z)
        return self.out_proj(y) + residual